코딩할 때 쓸만한 노트북 추천해줘" 라는 질문이 들어왔을 때 → 라우터가 분류 → LLM이 DB 데이터 기반으로 추천 → 결과 출력

환경 설정 (API 키 로드, LLM 클래스)  
데이터 준비 (상품 DB + 컨텍스트 문자열 생성)  
프롬프트 함수 (LLM 호출 + JSON 파싱)  
라우터 (키워드로 경로 결정)  
통합 실행 (라우터 → 도구 → 결과 출력)  

In [5]:
# 환경 설정 & LLM 클래스
import os 
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key ...')

class OpenAILLM:
    def __init__(self, model: str = 'gpt-4o-mini'):
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def generate(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            # 채팅 형식(system + user 메세지)으로 LLm에게 질문 보냄
            model=self.model,
            messages=[
                {"role": "system", "content" : "You are an ecommerce recommendation assistant, Return only valid JSON"},
                {"role": "user", "content": prompt}
            ],
            temperature = 0,
            response_format={'type': 'json_object'}
            # LLM이 반드시 JSON 형식으로만 응답하도록 강제
        )
        return response.choices[0].message.content
        # response.choices : 응답 후보 목록
        # .message.content : 실제 텍스트 내용 추출

llm = OpenAILLM() # 객체 생성
print(llm.model)

gpt-4o-mini


In [6]:
# 데이터 준비 (상품 DB + 컨텍스트 문자열 생성)

# 여기선 하드코딩으로 대체 (RAG에서 말하는 'context'에 해당)
do_search_results = [
    {"id": "p1", "name": "고성능 노트북",  "category": "가전"},
    {"id": "p2", "name": "사무용 랩탑",    "category": "가전"},
    {"id": "p3", "name": "미러리스 카메라", "category": "가전"},
]

# LLM에게 넘길 컨텍스트 문자열을 담을 빈 문자열 초기화
context_string = ""

for item in do_search_results:
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}\n"

print(context_string)

# LLM은 텍스트만 이해 => 읽기 편한 텍스트 문자열로 변환해서 전달

- 상품명:고성능 노트북 | 카테고리:가전
- 상품명:사무용 랩탑 | 카테고리:가전
- 상품명:미러리스 카메라 | 카테고리:가전



In [7]:
# 프롬프트 함수 (LLM호출 + JSON파싱)
from textwrap import dedent
# 들여쓰기 제거 도구 
# 함수 안에서 '''문자열'''을 쓰면 코드 들여쓰기만큼 앞에 공백이 자동으로 붙음

import json
# JSON 문자열 <-> 파이썬 딕셔너리 변환 담당
# json.loads() : JSON 문자열 -> 파이썬 딕셔너리
# json.dumps() : 파이썬 딕셔너리 -> JSON 문자열

def recommend_product(user_question: str, context: str) -> dict:
    # user_question : 사용자의 질문 문자열
    # context : 2단계에서 만든 상품 목록 문자열 (LLM이 참고할 데이터)
    # -> dict : 최종적으로 딕셔너리를 반환

    prompt = dedent(f'''
                        당신은  사용자의 질문에 정확히 응답하는 ai 시스템 입니다.
                        사용자의 질문과 context를 보고 질문의 의도에 맞게 출력하세요

                        [컨텍스트]
                        {context}

                        [질문]
                        {user_question}                        
                        
                        [출력]
                        답변은 반드시 아래와 같은 json형태로 
                        {
                            {
                                "assistant" : "출력내용",
                                "reason":"사유"                            
                            }
                        }

                        ''')
    response = llm.generate(prompt)
    data = json.loads(response)
    return json.dumps(data, indent=2, ensure_ascii=False)
    # 파이썬 딕셔너리 -> JSON 문자열로 다시 변환해서 반환
    # indent=2 : 들여쓰기 2칸
    # ensure_ascii=False : 한글이 \uXXXX 유니코드로 깨지지 않고 그대로 출력되게 함

result = recommend_product("코딩할 때 쓸만한 노트북 추천해줘", context_string)
print(result)

{
  "assistant": "고성능 노트북을 추천합니다.",
  "reason": "코딩 작업에는 높은 성능과 처리 능력이 필요한데, 고성능 노트북이 이러한 요구를 충족시켜 줄 수 있습니다."
}


In [8]:
# 라우터 (키워드로 경로 설정)
def route_qeustion(question: str) -> str:
    # 사용자 질문을 받아서 어떤 도구로 보낼지 결정하는 함수

    lower_question = question.lower()
    # 대소문자 때문에 키워드 매칭이 실패하는 걸 방지

    if any(keyword in lower_question for keyword in ['뉴스', '기사', '검색', '찾아줘', '최신', '오늘']):
        return "news_tool"
        # any() : 리스트 안의 키워드 중 하나라도 질문에 포함되면 True
        # '뉴스', '기사' 등의 단어가 있으면 → 네이버 뉴스 검색 도구로 보냄

    elif any(keyword in lower_question for keyword in ['계산', '더하기', '곱하기', '합계', '몇', '얼마']):
        return "calculator_tool"
        # 계산 관련 키워드가 있으면 → 계산기 도구로 보냄

    elif any(keyword in lower_question for keyword in ['기억', '기록', '선호', '메모', '이전']):
        return "memory_tool"
        # 기억/저장 관련 키워드가 있으면 → 메모리 저장 도구로 보냄

    elif any(keyword in lower_question for keyword in ['추천', '골라', '비교']):
        return "llm_recommendation"
        # 추천 관련 키워드가 있으면 → LLM 추천 도구로 보냄

    else:
        return "general_llm"
        # 위 키워드 중 아무것도 없으면 → 일반 LLM 응답으로 보냄

# 테스트용 질문 목록
sample_questions = [
    "3개 상품을 2개씩 주문하면 총 몇 개인가?",   # → calculator_tool
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘.",  # → memory_tool
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘.",    # → news_tool
    "코딩할 때 쓸만한 노트북 추천해줘.",          # → llm_recommendation
]

for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_qeustion(question)}')
    # 각 질문이 어떤 도구로 분류되는지 확인

질문 : 3개 상품을 2개씩 주문하면 총 몇 개인가? | 라우트 : calculator_tool
질문 : 나는 코딩용 노트북을 좋아한다는 점을 기억해줘. | 라우트 : memory_tool
질문 : AI 에이전트 뉴스 최신 기사 3개 찾아줘. | 라우트 : news_tool
질문 : 코딩할 때 쓸만한 노트북 추천해줘. | 라우트 : llm_recommendation


In [9]:
# 통합 실행 (라우터 -> 도구 -> 결과 출력)
import re

# -------------------------------------------------------
# 도구 1 : 계산기
# -------------------------------------------------------
def calculator_tool(text: str) -> float:
    # 수식 문자열을 받아서 계산 결과를 반환하는 함수

    allowed_chars = set("0123456789+-*/(). ")
    # 계산식에 허용할 문자들의 집합
    # set으로 만드는 이유 : 리스트보다 in 검사가 빠름

    if not set(text) <= allowed_chars:
        raise ValueError("허용되지 않은 문자가 포함")
    # <= : 왼쪽 집합이 오른쪽 집합의 부분집합인지 확인

    return eval(text)
    # 문자열을 파이썬 코드로 실행해서 계산결과 반환

In [10]:
# -------------------------------------------------------
# 도구 2 : 네이버 뉴스 검색
# -------------------------------------------------------
import os
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(
        pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>', "", text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []

def search_naver_news(query: str, display: int =3) -> list[dict]:
    encText = urllib.parse.quote(query)
    # 한글 검색어를 URL에 쓸 수 있는 형식으로 인코딩

    encText += f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText

    request = urllib.request.Request(url)

    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)

    response = urllib.request.urlopen(request)

    rescode = response.getcode()

    if rescode == 200:
        response_body = response.read().decode('utf-8')

        result = json.loads(response_body)

        for row in result.get('items'):
            items.append({
                'title':   _format_str(row.get('title')),
                # 기사 제목 (HTML 태그 제거 후 저장)
                'content': _format_str(row.get('description')),
                # 기사 요약 내용 (HTML 태그 제거 후 저장)
                'date':    _format_date(row.get('pubDate')),
                # 발행 날짜 (읽기 좋은 형식으로 변환 후 저장)
                'link':    row.get('link')
                # 기사 원문 링크
            })
    return items

In [11]:
# -------------------------------------------------------
# 도구 3 : 메모리
# -------------------------------------------------------
session_memory = {}
# 사용자 정보를 저장하는 딕셔너리

def remember_preference(user_id: str, key: str, value: str) -> None:
    # user_id : 사용자 식별자
    # key     : 저장할 정보의 이름 (예: 'category', 'usage')
    # value   : 저장할 값 (예: '노트북', '코딩')
    # -> None : 반환값 없음. 저장만 함

    if user_id not in session_memory:
        session_memory[user_id] = {}
        # 처음 저장하는 사용자면 빈 딕셔너리로 초기화

    session_memory[user_id][key] = value

def get_preference(user_id: str, key: str, default: str | None = None) -> str | None:
    # 저장된 사용자 선호 정보를 꺼내는 함수
    # default : 해당 정보 없을 때 기본값 None
    return session_memory.get(user_id, {}).get(key, default)
    # session_memory.get(user_id, {}) : user_id가 없으면 빈 딕셔너리 반환
    # .get(key, default)              : key가 없으면 default 반환
    # 두 번 .get()을 체이닝해서 KeyError 없이 안전하게 접근


In [12]:
# -------------------------------------------------------
# 질문에서 수식 추출
# -------------------------------------------------------
def extract_math_expression(question: str) -> str:
    match = re.search(r"[0-9\s\+\-\*\/\(\)\.]+", question)
    # 질문 문자열에서 숫자와 사칙연산 기호로 이루어진 패턴을 찾음
    # r"[0-9\s\+\-\*\/\(\)\.]+" : 숫자, 공백, +,-,*,/,(,),. 가 1개 이상 연속된 패턴
    if not match:
        raise ValueError("계산식을 찾을 수 없습니다.")
    
    expression = match.group(0).strip()
    # match.group(0) : 매칭된 문자열 전체를 가져옴
    
    if not expression:
        raise ValueError("계산식이 비어 있습니다.")
    
    return expression

In [13]:
# -------------------------------------------------------
# 질문에서 뉴스 검색 키워드 추출
# -------------------------------------------------------
def extract_news_query(question: str) -> str:
    cleaned = re.sub(r"뉴스|기사|검색|알려줘|찾아줘|추천해줘|좀|최근|최신|오늘", " ", question)
    # 검색에 불필요한 단어들을 공백으로 제거
    # "AI 에이전트 뉴스 최신 기사 3개 찾아줘" → "AI 에이전트    3개  "

    cleaned = re.sub(r"\d+\s*개?", " ", cleaned)

    cleaned = re.sub(r"[^0-9A-Za-z가-힣\s]", " ", cleaned)
    # 한글, 영문, 숫자, 공백 외의 특수문자를 공백으로 제거

    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    # 여러 개의 공백을 하나로 합치고 앞뒤 공백 제거
    # "AI  에이전트   " → "AI 에이전트"

    return cleaned or question
# cleaned가 빈 문자열이면 원래 question을 그대로 반환


In [15]:
# -------------------------------------------------------
# 통합 실행 함수 : 모든 것을 하나로 연결
# -------------------------------------------------------
def route_and_execute(question: str) -> dict:
    # 사용자 질문을 받아서
    # 1. 라우터로 경로 결정
    # 2. 해당 도구 실행
    # 3. 결과를 딕셔너리로 반환

    route = route_qeustion(question)
    # 4단계에서 만든 라우터로 어떤 도구를 쓸지 결정

    if route == "news_tool":
        news_query = extract_news_query(question)
        # 질문에서 불필요한 단어 제거 후 핵심 키워드만 추출

        news_result = search_naver_news(news_query, display=3)
        # 추출한 키워드로 네이버 뉴스 검색

        news_result = recommend_product('뉴스 요약해줘', news_result)
        # 검색된 뉴스를 LLM에게 넘겨서 요약 생성
        # 3단계의 recommend_product를 재활용

        return {
            "route": route,            # 선택된 도구 이름
            "tool": "search_naver_news",  # 실제 실행된 함수 이름
            "input": news_query,       # 도구에 넣은 입력값
            "result": news_result,     # 도구 실행 결과
        }

    if route == "calculator_tool":
        expression = extract_math_expression(question)
        # 질문에서 계산식만 추출

        result = calculator_tool(expression)
        # 추출한 계산식을 계산기 도구로 실행

        return {
            "route": route,
            "tool": "calculator_tool",
            "input": expression,
            "result": result,
        }

    if route == "memory_tool":
        remember_preference("student-001", "last_question", question)
        # 질문을 메모리에 저장
        # 실제 서비스라면 로그인한 사용자 ID를 쓰겠지만 여기선 고정값 사용

        return {
            "route": route,
            "tool": "memory_tool",
            "input": question,
            "result": get_preference("student-001", "last_question"),
            # 방금 저장한 값을 바로 꺼내서 확인
        }

    if route == "llm_recommendation":
        recommendation = recommend_product(question, context_string)
        # 3단계의 recommend_product로 LLM 추천 생성
        # context_string : 2단계에서 만든 상품 목록

        return {
            "route": route,
            "tool": "llm_recommendation",
            "input": question,
            "result": recommendation,
        }

    return {
        "route": route,
        "tool": "general_llm",
        "input": question,
        "result": question,
        # 어떤 도구도 해당 안 되면 질문을 그대로 반환
        # 실제 서비스라면 여기서 일반 LLM 호출을 추가해야 함
    }


# -------------------------------------------------------
# 실행 테스트
# -------------------------------------------------------
result = route_and_execute('코딩할 때 쓸만한 노트북 추천해줘')
print(json.loads(result['result'])['assistant'])
print(json.loads(result['result'])['reason'])

고성능 노트북을 추천합니다.
코딩 작업에 필요한 성능과 처리 능력을 갖추고 있어 효율적인 작업이 가능합니다.
